### Imports + paths 

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

import duckdb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss


### Robust repo root + paths (fixes your DB_PATH issues)

In [2]:
# --- find repo root robustly (walk up until we see .git or "Day-11") ---
HERE = Path.cwd()

def find_repo_root(p: Path) -> Path:
    for parent in [p] + list(p.parents):
        if (parent / ".git").exists():
            return parent
        if (parent / "Day-11").exists():
            return parent
    return p

REPO_ROOT = find_repo_root(HERE)
DAY18_DIR = REPO_ROOT / "Day-18"
REPORTS = DAY18_DIR / "reports"
ARTIFACTS = DAY18_DIR / "artifacts"

REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)

DB_PATH = REPO_ROOT / "Day-11" / "data" / "warehouse" / "day11_noshow.duckdb"

print("HERE     :", HERE)
print("REPO_ROOT:", REPO_ROOT)
print("DB_PATH  :", DB_PATH)
print("DB exists:", DB_PATH.exists())


HERE     : C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\notebooks
REPO_ROOT: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science
DB_PATH  : C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-11\data\warehouse\day11_noshow.duckdb
DB exists: True


### Connect to DuckDB (read-only) + confirm table

In [3]:
try:
    con = duckdb.connect(str(DB_PATH), read_only=True)
    print("Connected ✅")
    print("Tables:", con.execute("SHOW TABLES").fetchall())
except Exception as e:
    print("❌ DuckDB open failed.")
    print("Most common cause: DB file is locked by another python/notebook.")
    print("Fix: close other notebooks/kernels using DuckDB, restart Jupyter, or kill the locking PID.")
    raise


Connected ✅
Tables: [('bronze_appointments',), ('gold_appointments_base',), ('gold_appointments_features_v1',), ('gold_appointments_features_v1_patient_split',), ('gold_appointments_splits',), ('silver_appointments',), ('split_patient_v1',)]


### Load features table (Day-11 gold)

In [4]:
FEATURES_TBL = "gold_appointments_features_v1"

df = con.execute(f"SELECT * FROM {FEATURES_TBL}").df()
con.close()

print("Loaded df shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()


Loaded df shape: (110516, 25)
Columns: ['appointment_id', 'person_id', 'label', 'sms_received', 'age', 'gender', 'neighbourhood', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'lead_time_days', 'lead_time_clipped', 'lead_time_log1p', 'lead_time_bin', 'appt_date', 'sched_date', 'appt_dow', 'appt_month', 'sched_dow', 'sched_month', 'sched_hour', 'nbhd_n', 'prior_appt_count']


,appointment_id,person_id,label,sms_received,age,gender,neighbourhood,scholarship,hipertension,diabetes,...,lead_time_bin,appt_date,sched_date,appt_dow,appt_month,sched_dow,sched_month,sched_hour,nbhd_n,prior_appt_count
0,5644713,53134623314371,0,0,2.0,M,SANTO ANDRÉ,0,0,0,...,same_day,2016-05-02,2016-05-02,1,5,1,5,5,2571,0
1,5651477,53151829685666,0,0,59.0,F,JESUS DE NAZARETH,0,1,0,...,same_day,2016-05-03,2016-05-03,2,5,2,5,4,2853,0
2,5729499,53151829685666,0,0,60.0,F,JESUS DE NAZARETH,0,1,0,...,same_day,2016-05-24,2016-05-24,2,5,2,5,3,2853,1
3,5668104,53151829685666,1,0,60.0,F,JESUS DE NAZARETH,0,1,0,...,15_30,2016-06-02,2016-05-06,4,6,5,5,4,2853,2
4,5668105,53151829685666,1,0,60.0,F,JESUS DE NAZARETH,0,1,0,...,15_30,2016-06-02,2016-05-06,4,6,5,5,4,2853,3


### Define treatment/outcome + clean

In [5]:
A_COL = "sms_received"   # treatment: 1 if SMS received
Y_COL = "label"          # outcome: verify meaning; we treat label=1 as "no-show"

df = df.dropna(subset=[A_COL, Y_COL]).copy()
df[A_COL] = df[A_COL].astype(int)
df[Y_COL] = df[Y_COL].astype(int)

print("A counts:", df[A_COL].value_counts().to_dict())
print("Y counts:", df[Y_COL].value_counts().to_dict())

df.groupby(A_COL)[Y_COL].agg(["mean","count"])


A counts: {0: 75035, 1: 35481}
Y counts: {0: 88205, 1: 22311}


,mean,count
sms_received,,
0,0.166949,75035
1,0.275753,35481


### Create patient-level splits (train/valid/test) without DuckDB writes

In [6]:
rng = np.random.default_rng(42)

people = np.array(sorted(df["person_id"].dropna().unique()))
rng.shuffle(people)

n = len(people)
n_train = int(0.70 * n)
n_valid = int(0.15 * n)

train_ids = set(people[:n_train])
valid_ids = set(people[n_train:n_train+n_valid])
test_ids  = set(people[n_train+n_valid:])

def assign_split(pid):
    if pid in train_ids: return "train"
    if pid in valid_ids: return "valid"
    return "test"

df["split"] = df["person_id"].map(assign_split)

print(df["split"].value_counts())


split
train    77147
test     16768
valid    16601
Name: count, dtype: int64


### Choose covariates (simple + safe)

In [7]:
# Drop ids, treatment, outcome, and any date columns if present
drop_cols = {"appointment_id", "person_id", A_COL, Y_COL, "split"}
date_like = [c for c in df.columns if "date" in c.lower()]  # appt_date/sched_date if present
drop_cols = drop_cols.union(date_like)

X_cols = [c for c in df.columns if c not in drop_cols]

# Treat these as categorical even if numeric-coded
force_cat = [c for c in ["gender","neighbourhood","lead_time_bin","appt_dow","appt_month","sched_dow","sched_month","sched_hour"] if c in X_cols]

cat_cols = [c for c in X_cols if (df[c].dtype == "object") or (c in force_cat)]
num_cols = [c for c in X_cols if c not in cat_cols]

print("X_cols:", len(X_cols))
print("Categorical:", cat_cols)
print("Numeric:", num_cols)


X_cols: 19
Categorical: ['gender', 'neighbourhood', 'lead_time_bin', 'appt_dow', 'appt_month', 'sched_dow', 'sched_month', 'sched_hour']
Numeric: ['age', 'scholarship', 'hipertension', 'diabetes', 'alcoholism', 'handcap', 'lead_time_days', 'lead_time_clipped', 'lead_time_log1p', 'nbhd_n', 'prior_appt_count']


### Helper: build a fresh pipeline each time (prevents feature mismatch)

In [8]:
def make_logit_pipeline(cat_cols, num_cols, max_iter=4000):
    preprocess = ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler(with_mean=False))
            ]), num_cols),
            ("cat", Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore"))
            ]), cat_cols),
        ],
        remainder="drop"
    )
    clf = LogisticRegression(max_iter=max_iter, solver="saga")  # good for sparse OHE
    return Pipeline(steps=[("prep", preprocess), ("clf", clf)])


### Propensity model (optional but useful sanity check)

In [9]:
X = df[X_cols].copy()
A = df[A_COL].to_numpy()

m_tr = df["split"].eq("train").to_numpy()
m_va = df["split"].eq("valid").to_numpy()

prop_pipe = make_logit_pipeline(cat_cols, num_cols, max_iter=4000)
prop_pipe.fit(X[m_tr], A[m_tr])

ps_va = prop_pipe.predict_proba(X[m_va])[:, 1]
auc = roc_auc_score(A[m_va], ps_va)

print("Propensity ROC-AUC (train->valid):", round(auc, 4))


Propensity ROC-AUC (train->valid): 0.9146


### Uplift via T-learner (two outcome models, correctly separated)

In [10]:
Y = df[Y_COL].to_numpy()
m_te = df["split"].eq("test").to_numpy()

# IMPORTANT: fresh pipelines (no shared preprocessors)
y1_pipe = make_logit_pipeline(cat_cols, num_cols, max_iter=5000)
y0_pipe = make_logit_pipeline(cat_cols, num_cols, max_iter=5000)

y1_pipe.fit(X[m_tr & (A==1)], Y[m_tr & (A==1)])
y0_pipe.fit(X[m_tr & (A==0)], Y[m_tr & (A==0)])

mu1_te = y1_pipe.predict_proba(X[m_te])[:, 1]  # P(no-show | do SMS)
mu0_te = y0_pipe.predict_proba(X[m_te])[:, 1]  # P(no-show | do no SMS)

uplift_te = mu0_te - mu1_te  # + means SMS reduces no-show (good)

print("Mean uplift on TEST (mu0-mu1):", float(uplift_te.mean()))
print("Min/median/max uplift:", float(uplift_te.min()), float(np.median(uplift_te)), float(uplift_te.max()))


Mean uplift on TEST (mu0-mu1): -0.05311580499956223
Min/median/max uplift: -0.5933944623434453 -0.006734309800466992 0.33300283285458343


### Plot uplift distribution

In [11]:
plt.figure()
plt.hist(uplift_te, bins=60)
plt.xlabel("Uplift = mu0 - mu1  (positive = SMS helps)")
plt.ylabel("Count")
plt.title("Estimated Uplift Distribution (Test)")
plt.savefig(REPORTS / "DAY18_uplift_hist.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", REPORTS / "DAY18_uplift_hist.png")


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_uplift_hist.png


### Policy simulation: none vs all vs risk-based vs uplift-based

In [12]:
df_te = df.loc[m_te, ["appointment_id","person_id",A_COL,Y_COL]].copy().reset_index(drop=True)
df_te["mu0"] = mu0_te
df_te["mu1"] = mu1_te
df_te["uplift"] = uplift_te

def simulate_policy(df_te, k, policy):
    # policy: "none", "all", "risk", "uplift"
    n = len(df_te)
    treat = np.zeros(n, dtype=int)

    if policy == "none":
        treat[:] = 0
    elif policy == "all":
        treat[:] = 1
    elif policy == "risk":
        # treat highest baseline risk (mu0) first
        idx = np.argsort(-df_te["mu0"].to_numpy())[:k]
        treat[idx] = 1
    elif policy == "uplift":
        # treat highest uplift first
        idx = np.argsort(-df_te["uplift"].to_numpy())[:k]
        treat[idx] = 1
    else:
        raise ValueError("Unknown policy")

    # predicted no-show probability under policy:
    pred = np.where(treat==1, df_te["mu1"].to_numpy(), df_te["mu0"].to_numpy())
    return float(pred.mean())

budgets = [0.01, 0.05, 0.10, 0.20]
rows = []
n = len(df_te)

none_rate = simulate_policy(df_te, k=0, policy="none")
all_rate  = simulate_policy(df_te, k=n, policy="all")

for frac in budgets:
    k = int(round(frac * n))
    risk_rate = simulate_policy(df_te, k=k, policy="risk")
    upl_rate  = simulate_policy(df_te, k=k, policy="uplift")

    rows.append({
        "budget_frac": frac,
        "k": k,
        "no_show_rate_none": none_rate,
        "no_show_rate_all": all_rate,
        "no_show_rate_risk_policy": risk_rate,
        "no_show_rate_uplift_policy": upl_rate,
        "reduction_vs_none_risk": none_rate - risk_rate,
        "reduction_vs_none_uplift": none_rate - upl_rate,
    })

policy_tbl = pd.DataFrame(rows)
policy_tbl


,budget_frac,k,no_show_rate_none,no_show_rate_all,no_show_rate_risk_policy,no_show_rate_uplift_policy,reduction_vs_none_risk,reduction_vs_none_uplift
0,0.01,168,0.220366,0.273482,0.219062,0.218313,0.001304,0.002053
1,0.05,838,0.220366,0.273482,0.215497,0.212565,0.004869,0.007801
2,0.10,1677,0.220366,0.273482,0.211671,0.207007,0.008695,0.013358
3,0.20,3354,0.220366,0.273482,0.205516,0.198668,0.014850,0.021697


### Save tables + plot budget curves

In [13]:
policy_tbl.to_csv(REPORTS / "DAY18_policy_comparison.csv", index=False)

plt.figure()
plt.plot(policy_tbl["budget_frac"], policy_tbl["reduction_vs_none_risk"], marker="o")
plt.plot(policy_tbl["budget_frac"], policy_tbl["reduction_vs_none_uplift"], marker="o")
plt.xlabel("Budget fraction treated")
plt.ylabel("Predicted reduction in no-show rate vs NONE")
plt.title("Risk-based vs Uplift-based Targeting (Test)")
plt.savefig(REPORTS / "DAY18_budget_curve.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved:", REPORTS / "DAY18_policy_comparison.csv")
print("Saved:", REPORTS / "DAY18_budget_curve.png")


Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_policy_comparison.csv
Saved: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_budget_curve.png


### Export ranked lists (Top-K for each policy)

In [14]:
def export_ranked(df_te, frac, outname, score_col):
    k = int(round(frac * len(df_te)))
    out = df_te.sort_values(score_col, ascending=False).head(k).copy()
    out.to_csv(REPORTS / outname, index=False)
    return k, (REPORTS / outname)

for frac in [0.05, 0.10]:
    k1, p1 = export_ranked(df_te, frac, f"DAY18_top{int(frac*100)}pct_risk.csv", "mu0")
    k2, p2 = export_ranked(df_te, frac, f"DAY18_top{int(frac*100)}pct_uplift.csv", "uplift")
    print(f"Top {int(frac*100)}% (k={k1}) risk list ->", p1)
    print(f"Top {int(frac*100)}% (k={k2}) uplift list ->", p2)


Top 5% (k=838) risk list -> C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_top5pct_risk.csv
Top 5% (k=838) uplift list -> C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_top5pct_uplift.csv
Top 10% (k=1677) risk list -> C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_top10pct_risk.csv
Top 10% (k=1677) uplift list -> C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18_top10pct_uplift.csv


### Write DAY18.md automatically

In [15]:
md = f"""# Day 18 — Heterogeneous Effects / Uplift Modeling (SMS → No-Show)

## Goal
Estimate **who benefits most** from sending SMS reminders, and compare two targeting strategies under limited budgets:
1. **Risk-based**: send SMS to those with highest baseline predicted no-show risk (mu0).
2. **Uplift-based**: send SMS to those with largest estimated reduction in no-show probability (uplift = mu0 - mu1).

## Data + Setup
- Source: DuckDB table `{FEATURES_TBL}` built earlier (gold features).
- Treatment `A`: `{A_COL}` (1 = SMS received).
- Outcome `Y`: `{Y_COL}` (assumed 1 = no-show; confirm interpretation).
- Patient-level splits (by `person_id`): 70% train, 15% valid, 15% test.

## Method
We use a **T-learner**:
- Fit outcome model on treated: mu1(x) = P(Y=1 | X=x, A=1)
- Fit outcome model on control: mu0(x) = P(Y=1 | X=x, A=0)
- Uplift: uplift(x) = mu0(x) - mu1(x) (positive = SMS reduces no-show)

## Key Results (Test)
- Mean uplift (mu0 - mu1): {float(df_te["uplift"].mean()):.6f}
- Uplift distribution figure: `reports/DAY18_uplift_hist.png`
- Budget comparison table: `reports/DAY18_policy_comparison.csv`
- Budget curve: `reports/DAY18_budget_curve.png`

## Targeting Outputs
Ranked lists saved for operational use:
- Risk-based top lists: `reports/DAY18_top5pct_risk.csv`, `reports/DAY18_top10pct_risk.csv`
- Uplift-based top lists: `reports/DAY18_top5pct_uplift.csv`, `reports/DAY18_top10pct_uplift.csv`

## Interpretation
Risk targeting finds those likely to no-show, but **uplift targeting** aims to find those whose probability of no-show actually **changes most** under SMS. Under budget constraints, uplift-based targeting can outperform risk-based targeting when treatment effects are heterogeneous.
"""

out_path = REPORTS / "DAY18.md"
out_path.write_text(md, encoding="utf-8")
print("Wrote:", out_path)


Wrote: C:\Users\sarfo\Dropbox\Courses\Data Science\30-days-of-data-science\Day-18\reports\DAY18.md
